In [10]:
!pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 125.6 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.


In [21]:
import os
import numpy as np
import pandas as pd
import wfdb
from scipy.signal import butter, lfilter
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPool1D, Flatten, Dense, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ModelCheckpoint
import sklearn

def bandpass_filter(data, lowcut=0.001, highcut=15.0, signal_freq=360, filter_order=1):
    nyquist_freq = 0.5 * signal_freq
    low = lowcut / nyquist_freq
    high = highcut / nyquist_freq
    b, a = butter(filter_order, [low, high], btype="band")
    y = lfilter(b, a, data)
    return y

def load_challenge_data(filename):
    try:
        print(f"Attempting to load: {filename}")
        base_name = os.path.splitext(filename)[0]
        record = wfdb.rdsamp(base_name, return_res=16)
        if isinstance(record, tuple):
            data = np.asarray(record[0], dtype=np.float64)
            print(f"Loaded tuple data from {filename}, shape: {data.shape}")
        else:
            data = np.asarray(record.p_signal, dtype=np.float64)
            print(f"Loaded record data from {filename}, shape: {data.shape}")
        if data.ndim == 2 and data.shape[1] == 2:
            data = data.T
            print(f"Transposed data to: {data.shape}")
        elif data.ndim == 2 and data.shape[0] != 2:
            raise ValueError(f"Expected 2 leads, got shape {data.shape}")
        elif data.ndim == 1:
            print(f"Warning: Data is 1D with shape {data.shape}, attempting to reshape for 2 leads")
            if data.shape[0] % 2 != 0:
                raise ValueError(f"Cannot reshape 1D data of length {data.shape[0]} into 2 leads")
            num_samples = data.shape[0] // 2
            data = data.reshape(2, num_samples)
            print(f"Reshaped data to: {data.shape}")
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        raise
    new_file = filename.replace('.dat', '.hea')
    try:
        with open(new_file, 'r') as f:
            header_data = f.readlines()
        print(f"Loaded header from {new_file}")
    except Exception as e:
        print(f"Error loading header {new_file}: {e}")
        raise
    return data, header_data

def get_classes(input_directory, files):
    classes = ['N', 'L', 'R', 'A', 'S', 'V']
    return sorted(classes)

def get_true_labels(input_file, classes):
    atr_file = input_file.replace('.hea', '.atr')
    try:
        annotation = wfdb.rdann(input_file.replace('.hea', ''), 'atr')
        symbols = annotation.symbol
        print(f"Annotations for {atr_file}: {set(symbols)}")
        label_counts = {c: symbols.count(c) for c in classes}
        print(f"Label counts: {label_counts}")
        if not any(label_counts.values()):
            print(f"No matching annotations found in {atr_file}")
            return None, classes, np.zeros(len(classes), dtype=int)
        dominant_label = max(label_counts, key=label_counts.get)
        single_recording_labels = np.zeros(len(classes), dtype=int)
        if dominant_label in classes:
            idx = classes.index(dominant_label)
            single_recording_labels[idx] = 1
        print(f"Dominant label for {atr_file}: {dominant_label}")
        return dominant_label, classes, single_recording_labels
    except Exception as e:
        print(f"Error reading annotations from {atr_file}: {e}")
        return None, classes, np.zeros(len(classes), dtype=int)

def extend_ts(ts, length):
    extended = np.zeros(length)
    siglength = np.min([length, ts.shape[0]])
    extended[:siglength] = ts[:siglength]
    return extended

def compute_beta_score(labels, output, beta, num_classes, check_errors=True):
    if check_errors:
        if len(output) != len(labels):
            raise Exception('Numbers of outputs and labels must be the same.')
    num_recordings = len(labels)
    fbeta_l = np.zeros(num_classes)
    gbeta_l = np.zeros(num_classes)
    fmeasure_l = np.zeros(num_classes)
    accuracy_l = np.zeros(num_classes)
    f_beta = g_beta = f_measure = accuracy = 0
    C_l = np.ones(num_classes)
    for j in range(num_classes):
        tp = fp = fn = tn = 0
        for i in range(num_recordings):
            num_labels = np.sum(labels[i])
            if num_labels == 0:
                continue
            if labels[i][j] and output[i][j]:
                tp += 1 / num_labels
            elif not labels[i][j] and output[i][j]:
                fp += 1 / num_labels
            elif labels[i][j] and not output[i][j]:
                fn += 1 / num_labels
            elif not labels[i][j] and not output[i][j]:
                tn += 1 / num_labels
        if ((1 + beta**2) * tp + (fn * beta**2) + fp):
            fbeta_l[j] = float((1 + beta**2) * tp) / float(((1 + beta**2) * tp) + (fn * beta**2) + fp)
        else:
            fbeta_l[j] = 1.0
        if (tp + fp + beta * fn):
            gbeta_l[j] = float(tp) / float(tp + fp + beta * fn)
        else:
            gbeta_l[j] = 1.0
        if tp + fp + fn + tn:
            accuracy_l[j] = float(tp + tn) / float(tp + fp + fn + tn)
        else:
            accuracy_l[j] = 1.0
        if 2 * tp + fp + fn:
            fmeasure_l[j] = float(2 * tp) / float(2 * tp + fp + fn)
        else:
            fmeasure_l[j] = 1.0
    for i in range(num_classes):
        f_beta += fbeta_l[i] * C_l[i]
        g_beta += gbeta_l[i] * C_l[i]
        f_measure += fmeasure_l[i] * C_l[i]
        accuracy += accuracy_l[i] * C_l[i]
    f_beta = float(f_beta) / float(num_classes)
    g_beta = float(g_beta) / float(num_classes)
    f_measure = float(f_measure) / float(num_classes)
    accuracy = float(accuracy) / float(num_classes)
    return accuracy, f_measure, f_beta, g_beta

def create_model(frame_len, num_classes):
    model = Sequential([
        Input(shape=(frame_len, 2)),
        tf.keras.layers.GaussianNoise(0.1),
        Conv1D(64, 15, activation='relu'),
        MaxPool1D(2),
        Conv1D(64, 15, activation='relu'),
        MaxPool1D(2),
        Conv1D(64, 15, activation='relu'),
        MaxPool1D(2),
        Conv1D(64, 9, activation='relu'),
        MaxPool1D(3),
        Conv1D(64, 9, activation='relu'),
        MaxPool1D(3),
        Conv1D(32, 9, activation='relu'),
        MaxPool1D(3),
        Conv1D(32, 3, activation='relu'),
        MaxPool1D(4),
        Flatten(),
        Dense(64, kernel_initializer='normal', activation='relu'),
        Dense(num_classes, kernel_initializer='normal')
    ])
    model.compile(optimizer='adam',
                  loss='mean_squared_error',
                  metrics=['accuracy'])
    return model

if __name__ == '__main__':
    print(f"scikit-learn version: {sklearn.__version__}")
    input_directory = './mit-bih-arrhythmia-database-1.0.0/mit-bih-arrhythmia-database-1.0.0'
    num_leads = 2
    fs = 360
    frame_len = 15000
    num_folds = 5
    bs = 16
    ep = 2

    input_files = []
    for f in os.listdir(input_directory):
        if os.path.isfile(os.path.join(input_directory, f)) and not f.lower().startswith('.') and f.lower().endswith('.dat'):
            input_files.append(f)
    print("Found files:", input_files)
    if not input_files:
        raise ValueError(f"No .dat files found in {input_directory}")

    for f in input_files:
        dat_path = os.path.join(input_directory, f)
        hea_path = dat_path.replace('.dat', '.hea')
        atr_path = dat_path.replace('.dat', '.atr')
        print(f"Checking {f}: .dat={'Exists' if os.path.exists(dat_path) else 'Missing'}, "
              f".hea={'Exists' if os.path.exists(hea_path) else 'Missing'}, "
              f".atr={'Exists' if os.path.exists(atr_path) else 'Missing'}")

    classes = get_classes(input_directory, input_files)
    print("Classes:", classes)
    num_classes = len(classes)
    if num_classes == 0:
        raise ValueError("No classes found. Check .atr files for annotations.")

    num_files = len(input_files)
    X = np.zeros((num_files, frame_len, num_leads), dtype=np.float32)
    multi_labels = np.zeros((num_files, len(classes)), dtype=int)
    y = np.zeros((num_files), dtype=int)
    valid_files = 0

    for i, f in enumerate(input_files):
        print('    {}/{}...'.format(i + 1, num_files))
        tmp_input_file = os.path.join(input_directory, f)
        hea_file = tmp_input_file.replace('.dat', '.hea')
        if not os.path.exists(hea_file):
            print(f"Warning: .hea file not found for {f}: {hea_file}")
            continue
        try:
            data, header_data = load_challenge_data(tmp_input_file)
        except Exception as e:
            print(f"Skipping {f} due to error: {e}")
            continue

        if data.shape[0] != num_leads:
            print(f"Warning: Expected {num_leads} leads, got {data.shape[0]} for {f}")
            continue

        if data.shape[1] > frame_len:
            data = data[:, :frame_len]

        extended_data = np.zeros((num_leads, frame_len))
        for j in range(num_leads):
            if data[j, :].any():
                data[j, :] = (data[j, :] - np.expand_dims(data[j, :].mean(0), 0)) / np.expand_dims(data[j, :].std(0), 0)
            extended_data[j, :] = bandpass_filter(extend_ts(data[j, :], length=frame_len))

        X[i, :, :] = extended_data.T

        g = f.replace('.dat', '.hea')
        tmp_input_file = os.path.join(input_directory, g)
        recording_label, classes_label, multi_labels[i] = get_true_labels(tmp_input_file, classes)
        idx = np.where(multi_labels[i] == 1)
        if len(idx[0]) == 0:
            print(f"Warning: No valid label found for {f}")
            continue
        y[i] = idx[0][0]
        valid_files += 1

    print("Labels (y):", y)
    print(f"Unique labels: {np.unique(y)}")
    print(f"Valid files processed: {valid_files}")
    if valid_files == 0:
        raise ValueError("No valid files with labels extracted. Check .atr files or label extraction logic.")

    X = X[:valid_files]
    multi_labels = multi_labels[:valid_files]
    y = y[:valid_files]

    try:
        onehot_encoder = OneHotEncoder(sparse_output=False, categories=[np.arange(num_classes)])
    except TypeError:
        onehot_encoder = OneHotEncoder(sparse=False, categories=[np.arange(num_classes)])
    y_ = onehot_encoder.fit_transform(y.reshape(-1, 1))
    print(f"One-hot encoded labels shape: {y_.shape}")
    print('Done.')

    all_labels = np.zeros((valid_files, len(classes)), dtype=int)
    y = np.zeros((valid_files), dtype=int)

    label_files = []
    for f in os.listdir(input_directory):
        g = os.path.join(input_directory, f)
        if os.path.isfile(g) and not f.lower().startswith('.') and f.lower().endswith('.hea'):
            label_files.append(g)
    label_files = sorted(label_files)

    for k in range(valid_files):
        recording_label, classes_label, single_recording_labels = get_true_labels(label_files[k], classes)
        all_labels[k] = single_recording_labels
        idx = np.where(single_recording_labels == 1)
        if len(idx[0]) > 0:
            y[k] = idx[0][0]

    try:
        onehot_encoder = OneHotEncoder(sparse_output=False, categories=[np.arange(num_classes)])
    except TypeError:
        onehot_encoder = OneHotEncoder(sparse=False, categories=[np.arange(num_classes)])
    y_ = onehot_encoder.fit_transform(y.reshape(-1, 1))
    print(f"One-hot encoded labels shape (second loop): {y_.shape}")

    stats = np.zeros((num_folds, 4))
    kfold = KFold(n_splits=num_folds)

    fold_no = 1
    for id_train, id_test in kfold.split(X, y_):
        model = create_model(frame_len, num_classes)
        model_path = 'cnn_model3_fold_' + str(fold_no) + '.h5'
        checkpoint = ModelCheckpoint(model_path, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')

        print('------------------------------------------------------------------------')
        print(f'Training for fold {fold_no} ...')

        history = model.fit(X[id_train], y_[id_train], batch_size=bs, epochs=ep,
                           validation_data=(X[id_test], y_[id_test]),
                           callbacks=[checkpoint])

        accuracy = history.history['accuracy']
        val_accuracy = history.history['val_accuracy']
        loss = history.history['loss']
        val_loss = history.history['val_loss']
        epochs = range(len(accuracy))
        plt.figure()
        plt.plot(epochs, accuracy, label='Training accuracy')
        plt.plot(epochs, val_accuracy, label='Validation accuracy')
        plt.title('Training and validation accuracy')
        plt.legend()
        plt.savefig('accuracy_fold' + str(fold_no) + '.png')
        plt.close()

        plt.figure()
        plt.plot(epochs, loss, label='Training loss')
        plt.plot(epochs, val_loss, label='Validation loss')
        plt.title('Training and validation loss')
        plt.legend()
        plt.savefig('loss_fold' + str(fold_no) + '.png')
        plt.close()

        model.load_weights(model_path)

        print('------------------------------------------------------------------------')
        print('Calculating Confusion Matrix on train and Validation data')

        y_train_pred = model.predict(X[id_train])
        print(confusion_matrix(np.argmax(y_[id_train], 1), np.argmax(y_train_pred, 1)))

        y_test_pred = model.predict(X[id_test])
        print(confusion_matrix(np.argmax(y_[id_test], 1), np.argmax(y_test_pred, 1)))

        output = np.argmax(y_test_pred, 1)
        output_ = onehot_encoder.transform(output.reshape(-1, 1))

        _, f_measure, Fbeta_measure, Gbeta_measure = compute_beta_score(multi_labels[id_test], output_, 2, num_classes, check_errors=True)
        accuracy = max(val_accuracy)

        stats[fold_no - 1] = [accuracy, f_measure, Fbeta_measure, Gbeta_measure]

        print("Fold :", fold_no)
        print("Accuracy :", accuracy)
        print("F 1 :", f_measure)
        print("F_beta :", Fbeta_measure)
        print("G_beta :", Gbeta_measure)

        fold_no += 1

    df = pd.DataFrame(stats, columns=["Accuracy", "F_1", "F_beta", "G_beta"])
    df.to_csv('stats.csv')

scikit-learn version: 1.6.1
Found files: ['212.dat', '102.dat', '234.dat', '230.dat', '209.dat', '100.dat', '233.dat', '107.dat', '112.dat', '111.dat', '200.dat', '124.dat', '205.dat', '119.dat', '221.dat', '217.dat', '214.dat', '231.dat', '210.dat', '207.dat', '114.dat', '228.dat', '106.dat', '219.dat', '118.dat', '104.dat', '201.dat', '223.dat', '220.dat', '208.dat', '213.dat', '101.dat', '108.dat', '203.dat', '113.dat', '116.dat', '123.dat', '222.dat', '232.dat', '105.dat', '109.dat', '122.dat', '103.dat', '202.dat', '117.dat', '115.dat', '215.dat', '121.dat']
Checking 212.dat: .dat=Exists, .hea=Exists, .atr=Exists
Checking 102.dat: .dat=Exists, .hea=Exists, .atr=Exists
Checking 234.dat: .dat=Exists, .hea=Exists, .atr=Exists
Checking 230.dat: .dat=Exists, .hea=Exists, .atr=Exists
Checking 209.dat: .dat=Exists, .hea=Exists, .atr=Exists
Checking 100.dat: .dat=Exists, .hea=Exists, .atr=Exists
Checking 233.dat: .dat=Exists, .hea=Exists, .atr=Exists
Checking 107.dat: .dat=Exists, .hea=Ex

3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.3384 - loss: 0.1447 - val_accuracy: 0.8000 - val_loss: 0.2795
Epoch 2/2
2/3 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.7969 - loss: 0.2409 
Epoch 2: val_accuracy did not improve from 0.80000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - accuracy: 0.7932 - loss: 0.2035 - val_accuracy: 0.8000 - val_loss: 0.1101
------------------------------------------------------------------------
Calculating Confusion Matrix on train and Validation data
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 361ms/step
[[ 0  0  1  0]
 [ 0  0  3  0]
 [ 0  0 30  0]
 [ 0  0  4  0]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 347ms/step
[[0 1 0]
 [0 8 0]
 [0 1 0]]
Fold : 1
Accuracy : 0.800000011920929
F 1 : 0.4705882352941176
F_beta : 0.4868421052631579
G_beta : 0.45
------------------------------------------------------------------------
Training for fold 2 ...
Epoch 1/2
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2637 - loss: 0.1544  
Epoch 1: val_accuracy improved from -inf to 0.80000, savi

3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.3096 - loss: 0.1510 - val_accuracy: 0.8000 - val_loss: 0.2255
Epoch 2/2
2/3 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.7812 - loss: 0.1828
Epoch 2: val_accuracy did not improve from 0.80000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.7854 - loss: 0.1564 - val_accuracy: 0.8000 - val_loss: 0.0973
------------------------------------------------------------------------
Calculating Confusion Matrix on train and Validation data
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 368ms/step
[[ 0  0  1  0  0]
 [ 0  0  3  0  0]
 [ 0  0 30  0  0]
 [ 0  0  3  0  0]
 [ 0  0  1  0  0]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step
[[0 1 0]
 [0 8 0]
 [0 1 0]]
Fold : 2
Accuracy : 0.800000011920929
F 1 : 0.625
F_beta : 0.6470588235294118
G_beta : 0.6
------------------------------------------------------------------------
Training for fold 3 ...
Epoch 1/2
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3454 - loss: 0.1583  
Epoch 1: val_accuracy improved from -inf 

3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.3775 - loss: 0.1565 - val_accuracy: 0.8000 - val_loss: 0.0843
Epoch 2/2
2/3 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.7812 - loss: 0.0881
Epoch 2: val_accuracy did not improve from 0.80000
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.7854 - loss: 0.0838 - val_accuracy: 0.8000 - val_loss: 0.0615
------------------------------------------------------------------------
Calculating Confusion Matrix on train and Validation data
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 401ms/step
[[ 0  0  1  0  0]
 [ 0  0  3  0  0]
 [ 0  0 30  0  0]
 [ 0  0  3  0  0]
 [ 0  0  1  0  0]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step
[[0 1 0]
 [0 8 0]
 [0 1 0]]
Fold : 3
Accuracy : 0.800000011920929
F 1 : 0.8245614035087719
F_beta : 0.8297101449275361
G_beta : 0.8166666666666668
------------------------------------------------------------------------
Training for fold 4 ...
Epoch 1/2
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1271 - loss: 0.1621  
Epoch 1: val_

3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.1595 - loss: 0.1593 - val_accuracy: 0.7778 - val_loss: 0.1152
Epoch 2/2
2/3 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.7344 - loss: 0.1242 
Epoch 2: val_accuracy did not improve from 0.77778
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.7646 - loss: 0.1092 - val_accuracy: 0.7778 - val_loss: 0.0751
------------------------------------------------------------------------
Calculating Confusion Matrix on train and Validation data
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 367ms/step
[[ 0  0  1  0  0]
 [ 0  0  3  0  0]
 [ 0  0 31  0  0]
 [ 0  0  3  0  0]
 [ 0  0  1  0  0]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step
[[0 1 0]
 [0 7 0]
 [0 1 0]]
Fold : 4
Accuracy : 0.7777777910232544
F 1 : 0.823529411764706
F_beta : 0.8292682926829268
G_beta : 0.8148148148148149
------------------------------------------------------------------------
Training for fold 5 ...
Epoch 1/2
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6418 - loss: 0.1374   
Epoch 1: val

3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.6737 - loss: 0.1331 - val_accuracy: 0.7778 - val_loss: 0.0685
Epoch 2/2
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.7754 - loss: 0.0672 
Epoch 2: val_accuracy did not improve from 0.77778
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.7802 - loss: 0.0668 - val_accuracy: 0.7778 - val_loss: 0.0753
------------------------------------------------------------------------
Calculating Confusion Matrix on train and Validation data
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 397ms/step
[[ 0  4  0  0]
 [ 0 31  0  0]
 [ 0  3  0  0]
 [ 0  1  0  0]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step
[[0 1 0]
 [0 7 0]
 [0 1 0]]
Fold : 5
Accuracy : 0.7777777910232544
F 1 : 0.823529411764706
F_beta : 0.8292682926829268
G_beta : 0.8148148148148149
